# Mapping with Predefined Lists

In [1]:
import os, sys

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list

In [2]:
import importlib
importlib.reload(mapping_list)

<module 'List' from 'd:\\VS CODE\\pulse\\mapping\\List.py'>

In [3]:
def normalize_dataframe(df, column_variants):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    mapped_cols = {}
    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[std_col] = col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    df_extra = df.select(schema_cols + extra_cols)

    return new_df, df_extra, extra_cols, missing_cols, mapped_cols

In [4]:
base_dir = os.getcwd()
file_path_excel = os.path.join(base_dir, "./../faker/messy_inventory_data.xlsx")
file_path_csv = os.path.join(base_dir, "./../faker/messy_inventory_data.csv")

In [5]:
excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
excel_df.to_csv(file_path_csv, index=False)

In [6]:
spark = SparkSession.builder.appName("NormalizeData").getOrCreate()
df = spark.read.csv(file_path_csv, header=True, inferSchema=True)

In [7]:
df.show(5)

+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inv_id|product_ref|   vendor_id|current_stock|reserved_stock|min_stock_level|   last_restock_date|      last_sale_date|monthly_storage_cost|        created_date|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|100555|  PROD_0781|    SUPP_017|           72|            21|             69|2025-08-25 18:33:...|2025-09-07 03:54:...|                0.74|2024-05-31

In [8]:
new_df, extra_df, extra_cols, missing, mapped = normalize_dataframe(
    df, mapping_list.mapping_dict_inventory
)

In [9]:
print("\nNormalized DataFrame:")
new_df.show(5)

print("\nDataFrame with Extra Columns:")
extra_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_quantity|reorder_level|last_restocked_date|last_sold_date|storage_cost|created_at|
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|      100555| PROD_0781|    SUPP_017|            72|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100938| PROD_0373|    SUPP_003|             8|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100060| PROD_9999|  SUPP_029  |        Many  |             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100693| PROD_0244|    SUPP_022|         240  |             NULL|         NULL|               NULL|  

# Implementing NLTK

In [10]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.metrics.distance import edit_distance
from difflib import SequenceMatcher
import re

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#### Implementing Jaccard Similarity

In [11]:
def jaccard_similarity(source_column, target_column):
    source_col = source_column.lower()
    target_col = target_column.lower()
    intersection = len(set(source_col).intersection(set(target_col)))
    union = len(set(source_col).union(set(target_col)))
    jaccard_similarity = intersection / union if union > 0 else 0
    return jaccard_similarity

#### Implementing Sequence Matcher

In [12]:
def sequence_matching(source_column, target_column):
    source_col = source_column.lower()
    target_col = target_column.lower()
    return SequenceMatcher(None, source_col, target_col).ratio()

#### Implementing Edit Distance

In [13]:
def editing_distance(source_column, target_column):
    max_len = max(len(source_column), len(target_column))
    return 1 - (edit_distance(source_column.lower(), target_column.lower()) / max_len)

#### Combining them all

In [14]:
def mapping_with_combination(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            final = (
                0.4 * jaccard_similarity(missing_col, extra_col)
                + 0.3 * sequence_matching(missing_col, extra_col)
                + 0.3 * editing_distance(missing_col, extra_col)
            )
            print(f"{missing_col} -> {extra_col}: {final:.2f}")
            if final > best_score:
                best_score = final
                best_match = extra_col

        if best_match and best_score > threshold:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols


new_df, missing, extra_cols, mapped = mapping_with_combination(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

reserved_quantity -> reserved_stock: 0.55
reserved_quantity -> min_stock_level: 0.21
reserved_quantity -> last_restock_date: 0.29
reserved_quantity -> last_sale_date: 0.26
reserved_quantity -> monthly_storage_cost: 0.23
reserved_quantity -> created_date: 0.42
reserved_quantity -> available_qty: 0.37
reserved_quantity -> days_since_last_sale: 0.33
reserved_quantity -> stock_status: 0.26
reserved_quantity -> warehouse_location: 0.42
reserved_quantity -> total_stock_value: 0.18
reserved_quantity -> restock_lead_time_days: 0.43
reserved_quantity -> expiry_date: 0.33
reorder_level -> reserved_stock: 0.46
reorder_level -> min_stock_level: 0.41
reorder_level -> last_restock_date: 0.35
reorder_level -> last_sale_date: 0.27
reorder_level -> monthly_storage_cost: 0.23
reorder_level -> created_date: 0.35
reorder_level -> available_qty: 0.19
reorder_level -> days_since_last_sale: 0.23
reorder_level -> stock_status: 0.13
reorder_level -> warehouse_location: 0.32
reorder_level -> total_stock_value: 

#### Wordnet Mapping

In [15]:
def preprocess_column_name(column):
    name = re.sub("([A-Z][a-z]+)", r" \1", column)
    name = re.sub("_", " ", name)
    tokens = word_tokenize(name.lower())
    return [token for token in tokens if token.isalpha()]

In [16]:
def get_wordnet_synsets(word):
    return wordnet.synsets(word)

In [17]:
def calculate_semantic_similarity(missing_col, extra_col):
    missing_tokens = preprocess_column_name(missing_col)
    extra_tokens = preprocess_column_name(extra_col)
    if not missing_tokens or not extra_tokens:
        return 0.0
    max_similarities = []
    for token1 in missing_tokens:
        synsets1 = get_wordnet_synsets(token1)
        if not synsets1:
            continue
        token_similarities = []
        for token2 in extra_tokens:
            synsets2 = get_wordnet_synsets(token2)
            if not synsets2:
                continue
            similarities = [
                s1.path_similarity(s2)
                for s1 in synsets1
                for s2 in synsets2
                if s1.path_similarity(s2) is not None
            ]
            if similarities:
                token_similarities.append(max(similarities))
        if token_similarities:
            max_similarities.append(max(token_similarities))
    return sum(max_similarities) / len(max_similarities) if max_similarities else 0.0

In [18]:
def semantic_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.6):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            similarity = calculate_semantic_similarity(missing_col, extra_col)
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col
        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)
        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols
new_df, missing, extra_cols, mapped = semantic_column_mapping(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

Mapping: monthly_storage_cost -> storage_cost: 1.00

Normalized DataFrame:
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_stock|min_stock_level| last_restocked_date|      last_sale_date|storage_cost|          created_at|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|      100555| PROD_0781|    SUPP_017|            72|            21|             6

# Implementing ydata-profiling

In [19]:
from ydata_profiling import ProfileReport
pdf = df.toPandas()

profile = ProfileReport(pdf, title="Pandas Profiling Report", explorative=True)
profile.to_file("pandas_profiling_report.html")

c:\Users\fahad\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 29.11it/s]


In [22]:
desc = profile.description_set

def pandas_profiling_mapping(df,pdf, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = 0

        for extra_col in extra_cols[:]:
            if missing_col in pdf.columns and extra_col in pdf.columns:
                corr = abs(pdf[missing_col].corr(pdf[extra_col]))
            else:
                corr = 0
            if corr > best_score:
                best_score = corr
                best_match = extra_col

        if best_match and best_score > threshold:
            print(
                f"Data-based Mapping: {best_match} -> {missing_col} (corr={best_score:.2f})"
            )
            mapped_cols[missing_col] = best_match
            pdf = pdf.rename(columns={best_match: missing_col})
            extra_cols.remove(best_match)
            missing_cols.remove(missing_col)
        
    new_df = spark.createDataFrame(pdf)
    return new_df, missing_cols, extra_cols, mapped_cols
    

In [23]:
new_df, missing, extra_cols, mapped = pandas_profiling_mapping(df, pdf, missing, extra_cols, mapped, threshold=0.87)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inv_id|product_ref|   vendor_id|current_stock|reserved_stock|min_stock_level|   last_restock_date|      last_sale_date|monthly_storage_cost|        created_date|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|100555|  PROD_0781|    SUPP_017|           72|            21|             69|2025-08-25 18:33:...|2025-09-07 03:54:...|        